# Machine Learning Assignment - Text Classification

This notebook focuses on the code and explanation of the methods. It downloads public datasets directly and does not require mounting Google Drive. It does not create a report or a ZIP archive.

## Methods

1. **TF-IDF + traditional classifier:** learn vocabulary and TF-IDF weights from the fit split. Validation compares stop-word removal on/off, unigrams versus unigrams+bigrams, Multinomial Naive Bayes, Logistic Regression and linear SVM. Naive Bayes uses `alpha=1`; Logistic Regression and SVM try `C=1` and `C=4`.
2. **BGE-small + Logistic Regression:** use a pretrained, frozen Transformer to extract CLS embeddings and L2-normalize them. Validation chooses Logistic Regression `C` from `0.5`, `2` and `8`. The Transformer is not fine-tuned.
3. **TextCNN:** train word embeddings and convolutional classifier end-to-end from scratch using cross-entropy and Adam. Filters cover 3, 4 and 5 tokens; dropout is 0.5. The best epoch is selected by validation Macro-F1, with early stopping.

The original training split is divided into 85% fit and 15% validation; the original test set is kept for final evaluation. AG News is sampled to 10,000 train and 2,000 test instances; the other two use their official train/test splits. Exact duplicate text across splits is filtered out of the development sets and counted; source data and the test set are not modified. EDA covers class counts, text lengths and frequent words.

Public data: [AG News](https://huggingface.co/datasets/fancyzhx/ag_news), [Rotten Tomatoes](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes), [Banking77](https://huggingface.co/datasets/mteb/banking77). Model: [BGE-small](https://huggingface.co/BAAI/bge-small-en-v1.5).


## Step 1 - Install dependencies

Run the code cell below. Internet access is needed to install dependencies and download the datasets/model. A GPU runtime is recommended, but CPU is supported.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26', 'pandas>=2.1', 'matplotlib>=3.8', 'scipy>=1.12',
    'scikit-learn>=1.4', 'h5py>=3.10', 'datasets>=2.19',
    'transformers>=4.40', 'torch>=2.2'])
print('Dependency installation completed.')


## Step 2 - Load the pipeline

The current Python code is embedded in this notebook, so a fresh Colab session does not depend on an outdated GitHub checkout. Do not modify this cell manually.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
code_dir = root / '_notebook_code'
code_dir.mkdir(exist_ok=True)
(code_dir / 'assignment_pipeline.py').write_text('"""Reproducible text classification pipeline for the course assignment.\n\nFit and select on train/validation only. Evaluate each selected family once on\nheld-out test data. Artifacts are regenerated; historical reports are untouched.\n"""\nfrom __future__ import annotations\nimport json, random, re, time, platform\nfrom importlib.metadata import version\nfrom collections import Counter\nfrom pathlib import Path\n\nimport h5py\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.backends.backend_pdf import PdfPages\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.naive_bayes import MultinomialNB\nfrom sklearn.svm import LinearSVC\nfrom sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nSEED = 42\nREGISTRY = {\n \'ag_news\': (\'fancyzhx/ag_news\', 10000, 2000, 96),\n \'rotten_tomatoes\': (\'cornell-movie-review-data/rotten_tomatoes\', None, None, 64),\n \'banking77\': (\'mteb/banking77\', None, None, 64),\n}\nMODEL_ID=\'BAAI/bge-small-en-v1.5\'\nTOKEN = re.compile(r"[a-z0-9]+(?:\'[a-z]+)?")\nNEGATION = re.compile(r"\\b(not|no|never|nothing|nobody|none|cannot|nor|without)\\b|n\'t\\b", re.I)\n\ndef runtime_metadata():\n return {\'python\':platform.python_version(), **{n:version(n) for n in (\'numpy\',\'pandas\',\'scikit-learn\',\'scipy\',\'torch\',\'transformers\',\'datasets\',\'h5py\')}}\n\ndef seed_all(seed=SEED):\n import torch\n random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n\ndef words(s):\n return TOKEN.findall(str(s).lower())\n\ndef multiclass_brier(y, proba):\n """Mean over samples of sum_k (p_k - 1[y=k])^2."""\n y=np.asarray(y); proba=np.asarray(proba,dtype=np.float64)\n onehot=np.zeros_like(proba); onehot[np.arange(len(y)),y]=1.0\n return float(np.mean(np.sum((proba-onehot)**2,axis=1)))\n\ndef expected_calibration_error(y, proba, n_bins=15):\n """Top-label ECE over equal-width confidence bins."""\n proba=np.asarray(proba,dtype=np.float64); y=np.asarray(y)\n conf=proba.max(axis=1); correct=(proba.argmax(axis=1)==y).astype(float)\n edges=np.linspace(0,1,n_bins+1); ece=0.0\n for lo,hi in zip(edges[:-1],edges[1:]):\n  sel=(conf>lo)&(conf<=hi)\n  if sel.any(): ece+=sel.mean()*abs(correct[sel].mean()-conf[sel].mean())\n return float(ece)\n\ndef align_proba(classes, proba, n_classes):\n out=np.zeros((proba.shape[0],n_classes),dtype=np.float64)\n out[:,np.asarray(classes,dtype=int)]=proba\n return out\n\ndef metrics(y, pred, proba=None):\n p,r,f,_=precision_recall_fscore_support(y,pred,average=\'macro\',zero_division=0)\n out={\'accuracy\':accuracy_score(y,pred),\'precision_macro\':p,\'recall_macro\':r,\'f1_macro\':f,\n      \'brier_score\':np.nan,\'ece\':np.nan}\n if proba is not None:\n  proba=np.clip(np.asarray(proba,dtype=np.float64),0.0,1.0)\n  proba=proba/proba.sum(axis=1,keepdims=True).clip(min=1e-12)\n  out[\'brier_score\']=multiclass_brier(y,proba); out[\'ece\']=expected_calibration_error(y,proba)\n return out\n\ndef negation_mask(texts):\n return np.array([bool(NEGATION.search(str(t).lower())) for t in texts])\n\ndef negation_rows(family, texts, y, pred):\n has=negation_mask(texts); correct=(np.asarray(pred)==np.asarray(y))\n return {\'family\':family,\n         \'n_negation\':int(has.sum()),\'n_other\':int((~has).sum()),\n         \'acc_negation\':float(correct[has].mean()) if has.any() else np.nan,\n         \'acc_other\':float(correct[~has].mean()) if (~has).any() else np.nan,\n         \'gap\':float(correct[has].mean()-correct[~has].mean()) if has.any() and (~has).any() else np.nan}\n\ndef load_splits(name, quick=False):\n from datasets import load_dataset\n dataset_id, cap_train, cap_test, max_len = REGISTRY[name]\n raw = load_dataset(dataset_id)\n train, test = raw[\'train\'], raw[\'test\']\n quality={}\n for key,part in [(\'train\',train),(\'test\',test)]:\n  raw_text=part[\'text\']; valid=[str(t).strip() for t in raw_text if t is not None and str(t).strip()]\n  quality[key+\'_missing_or_blank\']=len(raw_text)-len(valid)\n  quality[key+\'_duplicate_texts\']=len(valid)-len(set(valid))\n names = list(train.features[\'label\'].names) if getattr(train.features[\'label\'], \'names\', None) else []\n if not names and \'label_text\' in train.column_names:\n  pairs = sorted({(int(i),str(s)) for i,s in zip(train[\'label\'],train[\'label_text\'])})\n  names = [s for _,s in pairs]\n def extract(split):\n  rows=[(str(t).strip(),int(y)) for t,y in zip(split[\'text\'],split[\'label\']) if t is not None and str(t).strip()]\n  return [a for a,_ in rows], np.array([b for _,b in rows],dtype=np.int64)\n def cap(text,y,n,seed):\n  if n is None or len(y)<=n: return text,y\n  ix,_=train_test_split(np.arange(len(y)),train_size=n,stratify=y,random_state=seed)\n  ix=np.sort(ix); return [text[i] for i in ix], y[ix]\n texts, labels=extract(train); texts,labels=cap(texts,labels, min(cap_train or 1500,1500) if quick else cap_train, SEED)\n tx,ty=extract(test); tx,ty=cap(tx,ty, min(cap_test or 600,600) if quick else cap_test, SEED+1)\n fit,val=train_test_split(np.arange(len(labels)),test_size=.15,stratify=labels,random_state=SEED)\n fit_set={texts[i] for i in fit}; val_set={texts[i] for i in val}; test_set=set(tx)\n quality.update({\'fit_val_exact_overlap_before\':len(fit_set & val_set),\'fit_test_exact_overlap_before\':len(fit_set & test_set),\'val_test_exact_overlap_before\':len(val_set & test_set)})\n original_fit,original_val=len(fit),len(val)\n val=np.array([i for i in val if texts[i] not in test_set],dtype=int)\n val_set={texts[i] for i in val}\n fit=np.array([i for i in fit if texts[i] not in test_set and texts[i] not in val_set],dtype=int)\n quality.update({\'removed_fit_for_overlap\':original_fit-len(fit),\'removed_val_for_overlap\':original_val-len(val),\n                 \'fit_test_exact_overlap\':len({texts[i] for i in fit} & test_set)})\n if set(labels[fit]) != set(labels) or set(labels[val]) != set(labels):\n  raise ValueError(\'Removing exact split overlaps left a class absent from fit or validation; revise the split.\')\n return {\'fit\':([texts[i] for i in fit],labels[fit]), \'val\':([texts[i] for i in val],labels[val]), \'test\':(tx,ty), \'names\':names, \'max_len\':max_len,\'source\':dataset_id,\'quality\':quality}\n\ndef write_sparse_h5(path, X, y):\n with h5py.File(path,\'w\') as f:\n  for k in (\'fit\',\'val\',\'test\'):\n   m=X[k].tocsr(); g=f.create_group(k); g.create_dataset(\'data\',data=m.data,compression=\'gzip\'); g.create_dataset(\'indices\',data=m.indices,compression=\'gzip\'); g.create_dataset(\'indptr\',data=m.indptr); g.attrs[\'shape\']=m.shape\n   f.create_dataset(\'y_\'+k,data=y[k])\n\ndef read_sparse_h5(path):\n with h5py.File(path,\'r\') as f:\n  return {k:sparse.csr_matrix((f[k][\'data\'][:],f[k][\'indices\'][:],f[k][\'indptr\'][:]),shape=tuple(f[k].attrs[\'shape\'])) for k in (\'fit\',\'val\',\'test\')}\n\ndef vectorizer(stop,ngram,tokenization=\'regex\'):\n extra={} if tokenization==\'regex\' else {\'tokenizer\':str.split,\'token_pattern\':None}\n return TfidfVectorizer(lowercase=True,stop_words=\'english\' if stop else None,ngram_range=(1,ngram),max_features=25000,min_df=2,sublinear_tf=True,dtype=np.float32,**extra)\n\ndef make_classifier(kind,c):\n if kind==\'nb\': return MultinomialNB(alpha=1.0)\n if kind==\'logreg\': return LogisticRegression(C=c,max_iter=1500,random_state=SEED)\n return LinearSVC(C=c,random_state=SEED,dual=\'auto\',max_iter=5000)\n\ndef select_sparse(split, feature_dir, tokenization=\'regex\', classifiers=(\'nb\',\'logreg\',\'svm\')):\n validation=[]; best=None\n n_classes=len(split.get(\'names\') or []) or int(max(split[\'fit\'][1].max(),split[\'val\'][1].max(),split[\'test\'][1].max())+1)\n for stop in (False,True):\n  for ng in (1,2):\n   vec=vectorizer(stop,ng,tokenization); fit=vec.fit_transform(split[\'fit\'][0]); val=vec.transform(split[\'val\'][0])\n   for model,c in [(m,c) for m in classifiers for c in ((1,) if m==\'nb\' else (1,4))]:\n    clf=make_classifier(model,c); t=time.perf_counter(); clf.fit(fit,split[\'fit\'][1]); pred=clf.predict(val)\n    proba=align_proba(clf.classes_,clf.predict_proba(val),n_classes) if hasattr(clf,\'predict_proba\') else None\n    row={\'family\':\'TF-IDF\',\'model\':model,\'stop_words\':stop,\'max_ngram\':ng,\'C\':c,\'fit_sec\':time.perf_counter()-t,**metrics(split[\'val\'][1],pred,proba)}\n    validation.append(row)\n    if best is None or row[\'f1_macro\']>best[0][\'f1_macro\']: best=(row,vec,clf)\n row,vec,clf=best\n X={\'fit\':vec.transform(split[\'fit\'][0]),\'val\':vec.transform(split[\'val\'][0]),\'test\':vec.transform(split[\'test\'][0])}\n y={k:split[k][1] for k in X}; path=feature_dir/\'tfidf.h5\'; write_sparse_h5(path,X,y)\n X=read_sparse_h5(path)\n clf=make_classifier(row[\'model\'],row[\'C\']); clf.fit(X[\'fit\'],y[\'fit\'])\n pred=clf.predict(X[\'test\'])\n proba=align_proba(clf.classes_,clf.predict_proba(X[\'test\']),n_classes) if hasattr(clf,\'predict_proba\') else None\n return validation, row, pred, proba, {\'vocabulary_size\':len(vec.vocabulary_), \'feature_file\':str(path)}\n\ndef encode_bge(split, feature_dir, padding, max_length, batch_size):\n import torch\n from transformers import AutoTokenizer, AutoModel\n tokenizer=AutoTokenizer.from_pretrained(MODEL_ID); device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n model=AutoModel.from_pretrained(MODEL_ID).to(device); model.eval(); outputs={}\n for k in (\'fit\',\'val\',\'test\'):\n  texts=split[k][0]; vectors=[]\n  with torch.inference_mode():\n   for start in range(0,len(texts),batch_size):\n    # padding can be dynamic or fixed; both use the same attention mask and truncation.\n    inp=tokenizer(texts[start:start+batch_size],padding=\'longest\' if padding==\'dynamic\' else \'max_length\',truncation=True,max_length=max_length,return_tensors=\'pt\')\n    inp={key:value.to(device) for key,value in inp.items()}\n    z=model(**inp).last_hidden_state[:,0]; z=torch.nn.functional.normalize(z,p=2,dim=1)\n    vectors.append(z.cpu().numpy().astype(\'float32\'))\n    if start == 0 or (start // batch_size + 1) % 50 == 0:\n     print(f\'  {k}: {min(start + batch_size, len(texts))}/{len(texts)}\',flush=True)\n  np.save(feature_dir/(\'bge_\'+k+\'.npy\'),np.concatenate(vectors))\n  outputs[k]=np.load(feature_dir/(\'bge_\'+k+\'.npy\'),mmap_mode=\'r\')\n del model\n if device.type==\'cuda\': torch.cuda.empty_cache()\n return outputs\n\ndef select_dense(split, X):\n validation=[]; best=None\n n_classes=len(split.get(\'names\') or []) or int(max(split[\'fit\'][1].max(),split[\'val\'][1].max(),split[\'test\'][1].max())+1)\n for c in (0.5,2.0,8.0):\n  model=make_pipeline(StandardScaler(),LogisticRegression(C=c,max_iter=1500,random_state=SEED))\n  t=time.perf_counter(); model.fit(X[\'fit\'],split[\'fit\'][1]); pred=model.predict(X[\'val\'])\n  proba=align_proba(model.classes_,model.predict_proba(X[\'val\']),n_classes)\n  row={\'family\':\'BGE-small\',\'model\':\'logreg\',\'C\':c,\'fit_sec\':time.perf_counter()-t,**metrics(split[\'val\'][1],pred,proba)}; validation.append(row)\n  if best is None or row[\'f1_macro\']>best[0][\'f1_macro\']: best=(row,model)\n pred=best[1].predict(X[\'test\']); proba=align_proba(best[1].classes_,best[1].predict_proba(X[\'test\']),n_classes)\n return validation,best[0],pred,proba\n\ndef train_textcnn(split, max_length, epochs, batch_size):\n import torch\n from torch import nn\n from torch.utils.data import TensorDataset,DataLoader\n seed_all(); counts=Counter(w for t in split[\'fit\'][0] for w in words(t)); vocab={w:i+2 for i,(w,n) in enumerate(counts.most_common(19998)) if n>=2}\n def encode(texts):\n  x=np.zeros((len(texts),max_length),dtype=np.int64)\n  for i,t in enumerate(texts):\n   ids=[vocab.get(w,1) for w in words(t)][:max_length]\n   if not ids: ids=[1]\n   x[i,:len(ids)]=ids\n  return torch.from_numpy(x)\n tensors={k:encode(split[k][0]) for k in (\'fit\',\'val\',\'test\')}\n class TextCNN(nn.Module):\n  def __init__(self):\n   super().__init__(); self.emb=nn.Embedding(len(vocab)+2,100,padding_idx=0); self.convs=nn.ModuleList([nn.Conv1d(100,64,k) for k in (3,4,5)]); self.dropout=nn.Dropout(.5); self.fc=nn.Linear(192,len(split[\'names\']))\n  def forward(self,x):\n   z=self.emb(x).transpose(1,2); outputs=[]\n   for conv in self.convs:\n    h=torch.relu(conv(z)); outputs.append(h.amax(dim=2))\n   return self.fc(self.dropout(torch.cat(outputs,dim=1)))\n device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\'); model=TextCNN().to(device); optimizer=torch.optim.Adam(model.parameters(),lr=.001); loss=nn.CrossEntropyLoss()\n def loader(k,shuffle=False):\n  return DataLoader(TensorDataset(tensors[k],torch.tensor(split[k][1])),batch_size=batch_size,shuffle=shuffle,generator=torch.Generator().manual_seed(SEED))\n def predict(k):\n  model.eval(); preds=[]; probs=[]\n  with torch.no_grad():\n   for x,_ in loader(k):\n    logits=model(x.to(device)); preds.extend(logits.argmax(1).cpu().tolist())\n    probs.append(torch.softmax(logits,dim=1).cpu().numpy())\n  return np.array(preds), np.concatenate(probs,axis=0)\n best=-1.; best_state=None; history=[]; stale=0\n for epoch in range(epochs):\n  model.train(); total=0\n  for x,y in loader(\'fit\',True):\n   x,y=x.to(device),y.to(device); optimizer.zero_grad(); l=loss(model(x),y); l.backward(); optimizer.step(); total+=l.item()*len(y)\n  pred,proba=predict(\'val\'); m=metrics(split[\'val\'][1],pred,proba); history.append({\'epoch\':epoch+1,\'train_loss\':total/len(tensors[\'fit\']),**m})\n  if m[\'f1_macro\']>best:\n   best=m[\'f1_macro\']; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0\n  else:\n   stale+=1\n   if stale>=2: break\n model.load_state_dict(best_state)\n pred,proba=predict(\'test\')\n return history,pred,proba,len(vocab)\n\ndef plot_eda(split,out,name):\n text,y=split[\'fit\']; lens=np.array([len(words(t)) for t in text]); freq=Counter(w for t in text for w in words(t)); top=freq.most_common(15)\n fig,axes=plt.subplots(1,3,figsize=(15,4)); axes[0].hist(lens,bins=30); axes[0].set_title(\'Train: words per text\'); axes[0].set_xlabel(\'Words\')\n counts=np.bincount(y,minlength=len(split[\'names\'])); axes[1].bar(range(len(counts)),counts); axes[1].set_title(\'Train class counts\'); axes[1].set_xlabel(\'Class ID\')\n axes[2].barh([w for w,_ in top][::-1],[n for _,n in top][::-1]); axes[2].set_title(\'Most frequent words\'); fig.suptitle(name); fig.tight_layout(); fig.savefig(out/\'eda.png\',dpi=140); plt.close(fig)\n return {\'n_fit\':len(text),\'n_val\':len(split[\'val\'][0]),\'n_test\':len(split[\'test\'][0]),\'classes\':len(counts),\'min_train_class\':int(counts.min()),\'max_train_class\':int(counts.max()),\'length_mean\':float(lens.mean()),\'length_p95\':float(np.percentile(lens,95)),\'vocabulary\':len(freq),\'top_words\':top}\n\ndef run_dataset(name,root,quick=False,epochs=7,batch_size=32,padding=\'dynamic\',tokenization=\'regex\',classifiers=(\'nb\',\'logreg\',\'svm\'),embeddings=(\'tfidf\',\'bge\')):\n seed_all(); root=Path(root); out=root/\'reports\'/name; features=root/\'features\'/name; out.mkdir(parents=True,exist_ok=True); features.mkdir(parents=True,exist_ok=True)\n split=load_splits(name,quick); print(f\'[{name}] fit/val/test:\',*(len(split[k][1]) for k in (\'fit\',\'val\',\'test\')),flush=True)\n eda=plot_eda(split,out,name); eda.update(split[\'quality\']); (out/\'eda.json\').write_text(json.dumps(eda,indent=2),encoding=\'utf8\')\n val=[]; rows=[]; neg=[]; info={}; history=[]\n if \'tfidf\' in embeddings:\n  val,choice,pred,proba,info=select_sparse(split,features,tokenization,classifiers)\n  rows.append({\'family\':\'TF-IDF\',\'selection\':json.dumps({k:choice[k] for k in (\'model\',\'stop_words\',\'max_ngram\',\'C\')}),**metrics(split[\'test\'][1],pred,proba)})\n  neg.append(negation_rows(\'TF-IDF\',split[\'test\'][0],split[\'test\'][1],pred)); np.save(out/\'tfidf_predictions.npy\',pred)\n np.save(features/\'y_fit.npy\',split[\'fit\'][1]); np.save(features/\'y_val.npy\',split[\'val\'][1]); np.save(features/\'y_test.npy\',split[\'test\'][1])\n if \'bge\' in embeddings:\n  print(f\'[{name}] BGE feature extraction\',flush=True); X=encode_bge(split,features,padding,split[\'max_len\'],batch_size)\n  v,choice,pred,proba=select_dense(split,X); val+=v\n  rows.append({\'family\':\'BGE-small\',\'selection\':json.dumps({\'model\':\'logreg\',\'C\':choice[\'C\']}),**metrics(split[\'test\'][1],pred,proba)})\n  neg.append(negation_rows(\'BGE-small\',split[\'test\'][0],split[\'test\'][1],pred)); np.save(out/\'bge_predictions.npy\',pred)\n print(f\'[{name}] TextCNN\',flush=True); history,pred,proba,vocab=train_textcnn(split,split[\'max_len\'],2 if quick else epochs,batch_size)\n rows.append({\'family\':\'TextCNN\',\'selection\':json.dumps({\'early_stopping\':\'validation macro-F1\',\'best_epoch\':max(history,key=lambda x:x[\'f1_macro\'])[\'epoch\'],\'vocab_size\':vocab}),**metrics(split[\'test\'][1],pred,proba)})\n neg.append(negation_rows(\'TextCNN\',split[\'test\'][0],split[\'test\'][1],pred)); np.save(out/\'textcnn_predictions.npy\',pred)\n pd.DataFrame(val).to_csv(out/\'validation.csv\',index=False)\n if history: pd.DataFrame(history).to_csv(out/\'textcnn_history.csv\',index=False)\n results=pd.DataFrame(rows); results.to_csv(out/\'test_results.csv\',index=False)\n pd.DataFrame(neg).to_csv(out/\'negation.csv\',index=False)\n (out/\'run_metadata.json\').write_text(json.dumps({\'dataset\':split[\'source\'],\'label_names\':split[\'names\'],\'seed\':SEED,\'quick\':quick,\'padding\':padding,\'tfidf_tokenization\':tokenization,\'classifiers\':list(classifiers),\'embeddings\':list(embeddings),\'max_length\':split[\'max_len\'],\'batch_size\':batch_size,\'epochs_limit\':(2 if quick else epochs),\'environment\':runtime_metadata(),\'bge_model\':MODEL_ID,\'features\':info,\'data\':eda},indent=2),encoding=\'utf8\')\n write_error_analysis(split,out)\n print(results.to_string(index=False),flush=True); return results\n\ndef write_error_analysis(split,out):\n """Save per-class statistics and examples for all selected model families."""\n out=Path(out); y=split[\'test\'][1]; names=split[\'names\']; labels=np.arange(len(names))\n present=set(pd.read_csv(out/\'test_results.csv\')[\'family\'])\n for family in (\'tfidf\',\'bge\',\'textcnn\'):\n  if {\'tfidf\':\'TF-IDF\',\'bge\':\'BGE-small\',\'textcnn\':\'TextCNN\'}[family] not in present: continue\n  path=out/(family+\'_predictions.npy\')\n  if not path.exists(): continue\n  pred=np.load(path)\n  report=classification_report(y,pred,labels=labels,target_names=names,output_dict=True,zero_division=0)\n  pd.DataFrame(report).T.to_csv(out/(family+\'_classification_report.csv\'))\n  errors=[{\'text\':split[\'test\'][0][i], \'true\':names[y[i]], \'predicted\':names[pred[i]]} for i in np.flatnonzero(pred!=y)[:30]]\n  pd.DataFrame(errors,columns=[\'text\',\'true\',\'predicted\']).to_csv(out/(family+\'_errors.csv\'),index=False)\n  matrix=confusion_matrix(y,pred,labels=labels)\n  np.save(out/(family+\'_confusion_matrix.npy\'),matrix)\n\ndef make_report(root,names,quick):\n import textwrap\n root=Path(root); reports=root/\'reports\'; rows=[]; neg_rows=[]\n for n in names:\n  p=reports/n/\'test_results.csv\'\n  if p.exists():\n   t=pd.read_csv(p); t.insert(0,\'dataset\',n); rows.append(t)\n  np_=reports/n/\'negation.csv\'\n  if np_.exists():\n   ntab=pd.read_csv(np_); ntab.insert(0,\'dataset\',n); neg_rows.append(ntab)\n if not rows: return\n data=pd.concat(rows,ignore_index=True); data.to_csv(reports/\'comparison.csv\',index=False)\n if neg_rows: pd.concat(neg_rows,ignore_index=True).to_csv(reports/\'negation_comparison.csv\',index=False)\n with PdfPages(reports/\'assignment_report.pdf\') as pdf:\n  fig=plt.figure(figsize=(8.27,11.69))\n  fig.text(.07,.95,\'MACHINE LEARNING ASSIGNMENT\',fontsize=17,weight=\'bold\')\n  fig.text(.07,.915,\'Task 2: Text classification\',fontsize=13)\n  lines=[\'HCMUT | Semester I, academic year 2026-2027\', \'\',\n   \'Course code: [team to complete]    Instructor: [team to complete]\',\n   \'Team members, IDs, email and contribution percentages: [team to complete]\',\n   \'Work distribution and collaboration evidence: [team to complete]\', \'\',\n   \'OBJECTIVE\',\n   \'Compare a conventional text pipeline, pretrained embeddings and a neural model.\', \'\',\n   \'EXPERIMENT DESIGN\',\n   \'Fit, validation and test are distinct. The fit set alone builds the TF-IDF vocabulary\',\n   \'and trains models. Validation macro-F1 chooses hyperparameters and early stopping.\',\n   \'Each family is evaluated once on the held-out official test set.\',\n   \'AG News: stratified sample of 10,000 training and 2,000 test records.\',\n   \'Rotten Tomatoes and Banking77: official train and test; train is split 85/15.\', \'\',\n   \'MODELS AND FEATURES\',\n   \'TF-IDF: stop-word removal on/off; unigrams or unigrams+bigrams; NB alpha=1;\',\n   \'logistic regression and linear SVM C in {1, 4} (20 validation configurations).\',\n   \'BGE-small: frozen pretrained encoder, CLS pooling; logistic regression C in {0.5, 2, 8}.\',\n   \'TextCNN: learned word embeddings and convolutions trained end-to-end from scratch;\',\n   \'Adam learning rate 0.001, dropout 0.5, up to 7 epochs, patience 2.\',\n   \'TF-IDF sparse matrices are stored as HDF5; BGE embeddings are stored as NPY.\',\n   \'The downstream classifiers read these stored features.\', \'\',\n   \'RELIABILITY AND LIMITATIONS\',\n   \'One random seed and one validation split; no confidence intervals are estimated.\',\n   \'The neural and traditional models have different architectures and training costs.\',\n   \'Padding settings use masks. Dataset selection must be confirmed with the instructor.\',\n   \'Do not use smoke-test metrics as final results. Review class-level errors before submission.\', \'\',\n   \'PUBLIC SOURCES\',\n   \'https://huggingface.co/datasets/fancyzhx/ag_news\',\n   \'https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes\',\n   \'https://huggingface.co/datasets/mteb/banking77\',\n   \'https://huggingface.co/BAAI/bge-small-en-v1.5\']\n  fig.text(.07,.875,\'\\n\'.join(lines),fontsize=9,va=\'top\',linespacing=1.5)\n  if quick: fig.text(.07,.04,\'SMOKE TEST — not final experimental results\',color=\'darkred\',fontsize=11,weight=\'bold\')\n  pdf.savefig(fig); plt.close(fig)\n  for name in names:\n   path=reports/name/\'run_metadata.json\'\n   if not path.exists(): continue\n   meta=json.loads(path.read_text(encoding=\'utf8\')); d=meta[\'data\']; subset=data[data.dataset==name]\n   fig=plt.figure(figsize=(8.27,11.69)); fig.text(.07,.95,f\'RESULTS: {name}\',fontsize=16,weight=\'bold\')\n   fig.text(.07,.921,f"Fit / validation / test: {d[\'n_fit\']} / {d[\'n_val\']} / {d[\'n_test\']}   |   classes: {d[\'classes\']}   |   seed: {SEED}",fontsize=9)\n   lines=[f"Fit-set length: mean {d[\'length_mean\']:.1f}, 95th percentile {d[\'length_p95\']:.1f} words; vocabulary {d[\'vocabulary\']}.",\n          f"Class counts in fit: minimum {d[\'min_train_class\']}, maximum {d[\'max_train_class\']}.",\n          f"Exact fit/test overlaps after filtering: {d.get(\'fit_test_exact_overlap\',\'not checked\')}; removed fit: {d.get(\'removed_fit_for_overlap\',\'?\')}, validation: {d.get(\'removed_val_for_overlap\',\'?\')}.", \'\',\n          \'Held-out test results: accuracy, macro precision, macro recall, macro F1\']\n   for row in subset.itertuples():\n    brier=getattr(row,\'brier_score\',np.nan); ece=getattr(row,\'ece\',np.nan)\n    lines.append(f\'{row.family:12s}  {row.accuracy:.4f}   {row.precision_macro:.4f}   {row.recall_macro:.4f}   {row.f1_macro:.4f}   Brier {brier:.4f}   ECE {ece:.4f}\')\n   neg_path=reports/name/\'negation.csv\'\n   if neg_path.exists():\n    lines+=[\'\',\'Negation vs other accuracy (cue words / n\\\'t):\']\n    for r in pd.read_csv(neg_path).itertuples():\n     lines.append(f\'{r.family:12s}  neg={r.acc_negation:.4f} (n={int(r.n_negation)})  other={r.acc_other:.4f} (n={int(r.n_other)})  gap={r.gap:.4f}\')\n   lines+=[\'\',\'Configurations selected on validation:\']\n   for row in subset.itertuples():\n    lines.extend(textwrap.wrap(f\'{row.family}: {row.selection}\',width=95))\n   ranking=subset.sort_values(\'f1_macro\',ascending=False)\n   if len(ranking)>1:\n    first,second=ranking.iloc[0],ranking.iloc[1]\n    lines += [\'\',f"In this run, {first[\'family\']} has the highest test macro-F1 ({first[\'f1_macro\']:.4f});",\n              f"{second[\'family\']} follows at {second[\'f1_macro\']:.4f}. This is descriptive, not model selection."]\n   lines+=[\'Results reflect a single split; inspect per-class CSVs and errors for limitations.\',\n           \'Training-only EDA below: length distribution, class counts and most frequent words.\']\n   fig.text(.07,.88,\'\\n\'.join(lines),fontsize=9,va=\'top\',linespacing=1.5)\n   img=plt.imread(reports/name/\'eda.png\'); ax=fig.add_axes([.055,.09,.89,.34]); ax.imshow(img); ax.axis(\'off\')\n   fig.text(.07,.045,\'Validation tables, confusion matrices and example errors accompany this report.\',fontsize=8)\n   if quick: fig.text(.67,.95,\'SMOKE TEST\',color=\'darkred\',fontsize=10,weight=\'bold\')\n   pdf.savefig(fig); plt.close(fig)\n\n # Editable companion report; no fabricated identities or experiment values.\n doc=[\'# Text classification: machine learning assignment\', \'\', \'HCMUT — Semester I, 2026–2027\', \'\', \'**Run:** \' + (\'SMOKE TEST — not final results\' if quick else \'Full configured datasets\'), \'\', \'## Team and work distribution\', \'\', \'| Name | Student ID | Email | Actual responsibility | Contribution |\', \'| --- | --- | --- | --- | --- |\', \'| [to complete] | [to complete] | [to complete] | [to complete] | [%] |\', \'| [to complete] | [to complete] | [to complete] | [to complete] | [%] |\', \'\', \'Instructor and course code: [to complete]. Percentages must total 100%. Add real collaboration evidence.\', \'\', \'## Objective and method\', \'\', \'Compare TF-IDF with traditional classifiers, frozen BGE-small embeddings with logistic regression, and end-to-end TextCNN. A stratified 85/15 split of the original training data is used for fitting and validation. The official test split is held out. AG News is stratified-sampled to 10,000 training and 2,000 test examples; Rotten Tomatoes and Banking77 use their original train/test sets. Rotten Tomatoes official validation is not used.\', \'\', \'The fit set alone is used to fit the TF-IDF vocabulary and classifiers. TF-IDF validation compares stop-word removal on/off, unigrams versus unigrams plus bigrams, Naive Bayes (alpha=1), and logistic regression or linear SVM (C=1 or 4): 20 configurations. BGE logistic regression compares C=0.5, 2 and 8. TextCNN uses 100-dimensional trainable embeddings, convolution widths 3/4/5, dropout 0.5, Adam learning rate 0.001, up to 7 epochs and patience 2. Validation macro-F1 selects TF-IDF configuration, BGE regularization and the TextCNN early-stopping epoch. One chosen configuration per family is evaluated on test. Frozen BGE-small is not end-to-end fine-tuning. Both dynamic and fixed padding use masks and identical maximum lengths.\', \'\', \'## Experimental results\', \'\']\n for name in names:\n  path=reports/name/\'run_metadata.json\'\n  if not path.exists(): continue\n  meta=json.loads(path.read_text(encoding=\'utf8\')); d=meta[\'data\']; sub=data[data.dataset==name]\n  doc += [f\'### {name}\', \'\', f"Dataset: `{meta[\'dataset\']}`. Fit / validation / test: {d[\'n_fit\']} / {d[\'n_val\']} / {d[\'n_test\']}; {d[\'classes\']} classes. Fit word count: mean {d[\'length_mean\']:.1f}, 95th percentile {d[\'length_p95\']:.1f}; vocabulary {d[\'vocabulary\']}. Fit class counts range from {d[\'min_train_class\']} to {d[\'max_train_class\']}.", \'\', f\'![Training-set EDA]({name}/eda.png)\', \'\', \'| Family | Accuracy | Macro P | Macro R | Macro F1 | Brier | ECE | Selected configuration |\', \'| --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |\']\n  doc.insert(len(doc)-4, f"Data quality: exact unique-text overlap fit/test before filtering = {d.get(\'fit_test_exact_overlap_before\',\'not checked\')}; removed fit records = {d.get(\'removed_fit_for_overlap\',\'not checked\')}, validation records = {d.get(\'removed_val_for_overlap\',\'not checked\')}. Missing/blank original train = {d.get(\'train_missing_or_blank\',\'not checked\')}, test = {d.get(\'test_missing_or_blank\',\'not checked\')}. Original source and test set are unchanged.")\n  for r in sub.itertuples():\n   brier=getattr(r,\'brier_score\',float(\'nan\')); ece=getattr(r,\'ece\',float(\'nan\'))\n   doc.append(f\'| {r.family} | {r.accuracy:.4f} | {r.precision_macro:.4f} | {r.recall_macro:.4f} | {r.f1_macro:.4f} | {brier:.4f} | {ece:.4f} | `{r.selection}` |\')\n  neg_path=reports/name/\'negation.csv\'\n  if neg_path.exists():\n   doc += [\'\', \'| Family | Acc (negation) | n | Acc (other) | n | Gap |\', \'| --- | ---: | ---: | ---: | ---: | ---: |\']\n   for r in pd.read_csv(neg_path).itertuples():\n    doc.append(f\'| {r.family} | {r.acc_negation:.4f} | {int(r.n_negation)} | {r.acc_other:.4f} | {int(r.n_other)} | {r.gap:.4f} |\')\n  doc += [\'\', f"Validation experiments: [`{name}/validation.csv`]({name}/validation.csv). TextCNN training history, per-class scores, confusion matrices and error examples are stored in the same folder.", \'\']\n  for family in (\'tfidf\',\'bge\',\'textcnn\'):\n   matrix_path=reports/name/(family+\'_confusion_matrix.npy\'); label_names=meta.get(\'label_names\',[])\n   if matrix_path.exists() and label_names:\n    matrix=np.load(matrix_path).copy(); np.fill_diagonal(matrix,0); a,b=np.unravel_index(np.argmax(matrix),matrix.shape)\n    if matrix[a,b]: doc.append(f"- {family}: the most frequent observed confusion is `{label_names[a]}` predicted as `{label_names[b]}` ({matrix[a,b]} examples). This is descriptive and may not represent other samples.")\n  ranked=sub.sort_values(\'f1_macro\',ascending=False)\n  if len(ranked)>1:\n   lead,runner=ranked.iloc[0],ranked.iloc[1]\n   doc += [\'\', f"In this held-out test, {lead[\'family\']} scored {lead[\'f1_macro\']:.4f} macro-F1 and {runner[\'family\']} scored {runner[\'f1_macro\']:.4f}; the observed gap is {lead[\'f1_macro\']-runner[\'f1_macro\']:.4f}. This is a descriptive single-run comparison, not evidence of a statistically significant difference or a basis for choosing hyperparameters."]\n  doc += [\'\', \'Review the saved examples and per-class metrics before drawing task-specific conclusions.\', \'\']\n doc += [\'## Metrics and interpretation\', \'\', \'Accuracy is the proportion of correct predictions. For each class, precision = TP / (TP + FP), recall = TP / (TP + FN), and F1 = 2 precision × recall / (precision + recall). Macro scores are the unweighted mean of per-class scores. Undefined class-level scores use zero. Macro-F1 is useful when class sizes differ because it gives each class equal weight.\', \'\', \'When class probabilities are available, the multiclass Brier score is the mean squared error between the predicted probability vector and the one-hot label; ECE is the top-label expected calibration error over equal-width confidence bins. Negation analysis reports accuracy on examples that match simple English negation cues versus the remainder.\', \'\', \'TF-IDF uses training-only document frequencies and weights a term by its within-document frequency and inverse document frequency. Linear classifiers learn decision boundaries on those sparse vectors. BGE-small supplies frozen pretrained contextual vectors; its logistic-regression classifier is trained on the fit split only. TextCNN learns its word vectors and convolution filters together using supervised cross-entropy and stops based on validation macro-F1.\', \'\', \'## Limitations and reproducibility\', \'\', \'The results use a single train/validation split and random seed, without confidence intervals. They do not establish causal explanations or generalization to new domains. Model and dataset versions can change; preserve the execution environment and feature files. Compare test results descriptively and use validation for configuration choices.\', \'\', \'## Sources\', \'\', \'- https://huggingface.co/datasets/fancyzhx/ag_news\', \'- https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes\', \'- https://huggingface.co/datasets/mteb/banking77\', \'- https://huggingface.co/BAAI/bge-small-en-v1.5\', \'\']\n (reports/\'assignment_report.md\').write_text(\'\\n\'.join(doc),encoding=\'utf8\')\n', encoding='utf-8')
sys.path.insert(0, str(code_dir))
from assignment_pipeline import REGISTRY, run_dataset
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Step 3 - Configure the experiment

The initial settings run a **quick smoke test on Rotten Tomatoes**. These results use a smaller sample and are only for checking whether the notebook works. Once that succeeds, set `QUICK = False` and select the datasets required by the group.

- TF-IDF varies stop-word removal, n-grams and the selected classifiers; `TOKENIZATION` controls its tokenizer.
- Both padding options use attention masks and the same truncation length for BGE.
- Keep both TF-IDF and BGE in `EMBEDDINGS` for the assignment's traditional and modern features.


In [ ]:
QUICK = True
DATASETS = ['rotten_tomatoes']  # options: ag_news, rotten_tomatoes, banking77
PADDING = 'dynamic'             # dynamic or fixed, for BGE
TOKENIZATION = 'regex'          # regex or whitespace, for TF-IDF
CLASSIFIERS = ['nb', 'logreg', 'svm']
EMBEDDINGS = ['tfidf', 'bge']
BATCH_SIZE = 32
EPOCHS = 7                     # quick mode uses 2 epochs
OUTPUT = root / 'experiment_outputs'


## Step 4 - Train and evaluate

The pipeline performs EDA, selects parameters on validation, evaluates the selected configuration from each family on the test set, and saves extracted features. It may take several minutes; a full run can take considerably longer.

In [ ]:
for name in DATASETS:
    run_dataset(name, OUTPUT, quick=QUICK, epochs=EPOCHS,
                batch_size=BATCH_SIZE, padding=PADDING,
                tokenization=TOKENIZATION, classifiers=CLASSIFIERS,
                embeddings=EMBEDDINGS)
print('Pipeline completed. Output folder:', OUTPUT)


## Step 5 - Inspect results

The first table reports held-out test metrics. The second table contains validation configurations. Macro-F1 gives each class equal weight. Inspect EDA and class-level errors before drawing conclusions.

In [ ]:
import pandas as pd
from IPython.display import display, Image
for name in DATASETS:
    print('Dataset:', name)
    display(pd.read_csv(OUTPUT / 'reports' / name / 'test_results.csv'))
    display(pd.read_csv(OUTPUT / 'reports' / name / 'validation.csv'))
    display(Image(filename=str(OUTPUT / 'reports' / name / 'eda.png')))
print('Features:', OUTPUT / 'features')
print('Metrics and analysis:', OUTPUT / 'reports')


## Output files

The Files panel on the left contains `experiment_outputs/features/` with `.h5` and `.npy` feature files and `experiment_outputs/reports/` with metrics, EDA figures and error-analysis CSVs. These files are in the temporary Colab runtime: download any files you need before the runtime is disconnected. The report will be completed separately later.